# PrototypeNine-v1.5 — YOLO Detection Comparison with Matched Random Baseline

This notebook builds **YOLO detection** datasets from the existing synthetic outputs and compares:

1. **Full / unfiltered dataset**
2. **Balanced VLM image-filtered dataset**
3. **Matched-size random subset of the full dataset**

The matched random subset is built to have the **same number of train and validation images** as the balanced filtered dataset. This helps separate the effect of **less data** from the effect of **quality-based filtering**.

It assumes you already ran:
- `PrototypeNine-v1_5_Generation_QA_Masks_Metadata.ipynb`
- `PrototypeNine-v1_5_VLM_QA_Filtering_LOCAL_v3_balanced.ipynb`

Expected folders under `output/`:
- `PrototypeNine_v1_5/`
- `PrototypeNine_v1_5_vlm_filtered_local_v3_balanced/`


In [ ]:
# -----------------------
# Section 0 - Imports + config
# -----------------------
import os, shutil, random
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm import tqdm

BASE_DIR = Path.cwd()
EXPERIMENT_NAME = "PrototypeNine_v1_5"

# Source datasets
full_root = BASE_DIR / "output" / EXPERIMENT_NAME
filtered_root = BASE_DIR / "output" / f"{EXPERIMENT_NAME}_vlm_filtered_local_v3_balanced"

# Input subfolders
full_image_dir   = full_root / "images"
full_label_dir   = full_root / "labels_yolo"

filtered_image_dir = filtered_root / "images"
filtered_label_dir = filtered_root / "labels_yolo"

# Output detection dataset roots
det_root_full    = BASE_DIR / "output" / f"{EXPERIMENT_NAME}_detect_full"
det_root_filtered = BASE_DIR / "output" / f"{EXPERIMENT_NAME}_detect_filtered_local_v3_balanced"
det_root_matched = BASE_DIR / "output" / f"{EXPERIMENT_NAME}_detect_full_matched_random"

# Split + training config
SEED = 42
MATCH_SEED = 1337
VAL_FRACTION = 0.20
USE_SYMLINKS = False   # safer on mixed environments; set True on Linux if you prefer
CLEAR_EXISTING_DATASET_DIRS = True

# Model/training config
YOLO_MODEL = "yolo11n.pt"   # change if you want a larger model
IMG_SIZE = 640
EPOCHS = 50
BATCH = 16                  # lower if you hit VRAM issues
DEVICE = 0                  # 0 for first GPU, 'cpu' if needed
PATIENCE = 20
WORKERS = 8
PROJECT_NAME = "rd2_yolo_detect_compare_matched"

# Class mapping used in the synthetic pipeline
CLASS_NAMES = ["rope", "plastic_bottles", "tires"]

print("[INFO] full_root:", full_root)
print("[INFO] filtered_root:", filtered_root)
print("[INFO] det_root_full:", det_root_full)
print("[INFO] det_root_filtered:", det_root_filtered)
print("[INFO] det_root_matched:", det_root_matched)

assert full_image_dir.exists(), f"Missing {full_image_dir}"
assert full_label_dir.exists(), f"Missing {full_label_dir}"
assert filtered_image_dir.exists(), f"Missing {filtered_image_dir}"
assert filtered_label_dir.exists(), f"Missing {filtered_label_dir}"
print("[INFO] Input folders found.")


In [ ]:
# -----------------------
# Section 1 - Helper functions
# -----------------------
def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def safe_link_or_copy(src: Path, dst: Path, use_symlinks: bool = False):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    if use_symlinks:
        try:
            os.symlink(src.resolve(), dst)
            return
        except Exception:
            pass
    shutil.copy2(src, dst)

def yolo_label_has_boxes(label_path: Path) -> bool:
    if not label_path.exists():
        return False
    txt = label_path.read_text(encoding="utf-8").strip()
    if not txt:
        return False
    return any(line.strip() for line in txt.splitlines())

def list_image_stems(image_dir: Path):
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    stems = []
    for p in sorted(image_dir.iterdir()):
        if p.suffix.lower() in exts:
            stems.append(p.stem)
    return stems

def find_image_for_stem(image_dir: Path, stem: str):
    for ext in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
        p = image_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def build_split_from_full(full_stems, val_fraction=0.2, seed=42):
    rng = random.Random(seed)
    stems = list(full_stems)
    rng.shuffle(stems)
    n_val = max(1, int(round(len(stems) * val_fraction))) if len(stems) > 1 else len(stems)
    val_set = set(stems[:n_val])
    train_set = set(stems[n_val:])
    if not train_set and val_set:
        moved = next(iter(val_set))
        train_set.add(moved)
        val_set.remove(moved)
    return train_set, val_set

def sample_matched_subset(full_train_stems, full_val_stems, filtered_train_count, filtered_val_count, seed=1337):
    rng = random.Random(seed)
    train_list = sorted(full_train_stems)
    val_list = sorted(full_val_stems)
    if filtered_train_count > len(train_list) or filtered_val_count > len(val_list):
        raise ValueError("Filtered counts exceed available full split counts.")
    matched_train = set(rng.sample(train_list, filtered_train_count))
    matched_val = set(rng.sample(val_list, filtered_val_count))
    return matched_train, matched_val

def prepare_detection_dataset(source_image_dir: Path,
                              source_label_dir: Path,
                              out_root: Path,
                              train_stems: set,
                              val_stems: set,
                              use_symlinks: bool = False):
    if CLEAR_EXISTING_DATASET_DIRS:
        reset_dir(out_root)
    else:
        ensure_dir(out_root)

    for split in ["train", "val"]:
        ensure_dir(out_root / "images" / split)
        ensure_dir(out_root / "labels" / split)

    kept_counts = Counter()
    missing = []

    def copy_split(stems, split):
        for stem in tqdm(sorted(stems), desc=f"Prepare {out_root.name}:{split}"):
            img = find_image_for_stem(source_image_dir, stem)
            lbl = source_label_dir / f"{stem}.txt"
            if img is None or not lbl.exists() or not yolo_label_has_boxes(lbl):
                missing.append((split, stem))
                continue
            safe_link_or_copy(img, out_root / "images" / split / img.name, use_symlinks=use_symlinks)
            safe_link_or_copy(lbl, out_root / "labels" / split / lbl.name, use_symlinks=use_symlinks)
            kept_counts[split] += 1

    copy_split(train_stems, "train")
    copy_split(val_stems, "val")
    return kept_counts, missing

def write_dataset_yaml(dataset_root: Path, yaml_path: Path, class_names):
    txt = f"path: {dataset_root.as_posix()}\ntrain: images/train\nval: images/val\n\nnames:\n"
    for idx, name in enumerate(class_names):
        txt += f"  {idx}: {name}\n"
    yaml_path.write_text(txt, encoding="utf-8")

def summarize_label_counts(label_dir: Path):
    class_counter = Counter()
    image_count = 0
    box_count = 0
    for p in sorted(label_dir.glob("*.txt")):
        lines = [ln.strip() for ln in p.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not lines:
            continue
        image_count += 1
        for ln in lines:
            parts = ln.split()
            if not parts:
                continue
            try:
                cls = int(float(parts[0]))
                class_counter[cls] += 1
                box_count += 1
            except Exception:
                pass
    return image_count, box_count, class_counter


In [ ]:
# -----------------------
# Section 2 - Build fair train/val split from full dataset
# -----------------------
full_stems_all = set(list_image_stems(full_image_dir))
filtered_stems_all = set(list_image_stems(filtered_image_dir))

print("[INFO] Full dataset images:", len(full_stems_all))
print("[INFO] Filtered dataset images:", len(filtered_stems_all))

train_stems_full, val_stems_full = build_split_from_full(full_stems_all, val_fraction=VAL_FRACTION, seed=SEED)

# Reuse the same split stems but intersect with what remains after filtering.
train_stems_filtered = train_stems_full.intersection(filtered_stems_all)
val_stems_filtered   = val_stems_full.intersection(filtered_stems_all)

print("[INFO] Full split -> train:", len(train_stems_full), "val:", len(val_stems_full))
print("[INFO] Filtered split -> train:", len(train_stems_filtered), "val:", len(val_stems_filtered))

if len(train_stems_filtered) == 0 or len(val_stems_filtered) == 0:
    print("[WARN] Filtered dataset lost all images in one split.")
    print("[WARN] You may want to regenerate the split using only filtered stems or lower filtering strictness.")

matched_train_stems, matched_val_stems = sample_matched_subset(
    full_train_stems=train_stems_full,
    full_val_stems=val_stems_full,
    filtered_train_count=len(train_stems_filtered),
    filtered_val_count=len(val_stems_filtered),
    seed=MATCH_SEED,
)

print("[INFO] Matched random split -> train:", len(matched_train_stems), "val:", len(matched_val_stems))


In [ ]:
# -----------------------
# Section 3 - Prepare YOLO detection datasets
# -----------------------
counts_full, missing_full = prepare_detection_dataset(
    source_image_dir=full_image_dir,
    source_label_dir=full_label_dir,
    out_root=det_root_full,
    train_stems=train_stems_full,
    val_stems=val_stems_full,
    use_symlinks=USE_SYMLINKS,
)

counts_filtered, missing_filtered = prepare_detection_dataset(
    source_image_dir=filtered_image_dir,
    source_label_dir=filtered_label_dir,
    out_root=det_root_filtered,
    train_stems=train_stems_filtered,
    val_stems=val_stems_filtered,
    use_symlinks=USE_SYMLINKS,
)

counts_matched, missing_matched = prepare_detection_dataset(
    source_image_dir=full_image_dir,
    source_label_dir=full_label_dir,
    out_root=det_root_matched,
    train_stems=matched_train_stems,
    val_stems=matched_val_stems,
    use_symlinks=USE_SYMLINKS,
)

yaml_full = det_root_full / "dataset.yaml"
yaml_filtered = det_root_filtered / "dataset.yaml"
yaml_matched = det_root_matched / "dataset.yaml"
write_dataset_yaml(det_root_full, yaml_full, CLASS_NAMES)
write_dataset_yaml(det_root_filtered, yaml_filtered, CLASS_NAMES)
write_dataset_yaml(det_root_matched, yaml_matched, CLASS_NAMES)

print("[INFO] Full dataset prepared:", counts_full)
print("[INFO] Filtered dataset prepared:", counts_filtered)
print("[INFO] Matched random dataset prepared:", counts_matched)
print("[INFO] YAML full:", yaml_full)
print("[INFO] YAML filtered:", yaml_filtered)
print("[INFO] YAML matched:", yaml_matched)

if missing_full:
    print(f"[WARN] Missing/skipped full items: {len(missing_full)}")
if missing_filtered:
    print(f"[WARN] Missing/skipped filtered items: {len(missing_filtered)}")
if missing_matched:
    print(f"[WARN] Missing/skipped matched items: {len(missing_matched)}")


In [ ]:
# -----------------------
# Section 4 - Dataset summaries
# -----------------------
def summarize_dataset(root: Path):
    train_img_count = len(list((root / "images" / "train").glob("*")))
    val_img_count   = len(list((root / "images" / "val").glob("*")))
    train_lbl_count, train_box_count, train_cls = summarize_label_counts(root / "labels" / "train")
    val_lbl_count, val_box_count, val_cls       = summarize_label_counts(root / "labels" / "val")
    return {
        "root": str(root),
        "train_images": train_img_count,
        "val_images": val_img_count,
        "train_label_files": train_lbl_count,
        "val_label_files": val_lbl_count,
        "train_boxes": train_box_count,
        "val_boxes": val_box_count,
        "train_class_counts": dict(train_cls),
        "val_class_counts": dict(val_cls),
    }

summary_full = summarize_dataset(det_root_full)
summary_filtered = summarize_dataset(det_root_filtered)
summary_matched = summarize_dataset(det_root_matched)

display(pd.DataFrame([
    {
        "dataset": "full_unfiltered",
        "train_images": summary_full["train_images"],
        "val_images": summary_full["val_images"],
        "train_boxes": summary_full["train_boxes"],
        "val_boxes": summary_full["val_boxes"],
    },
    {
        "dataset": "filtered_balanced",
        "train_images": summary_filtered["train_images"],
        "val_images": summary_filtered["val_images"],
        "train_boxes": summary_filtered["train_boxes"],
        "val_boxes": summary_filtered["val_boxes"],
    },
    {
        "dataset": "full_matched_random",
        "train_images": summary_matched["train_images"],
        "val_images": summary_matched["val_images"],
        "train_boxes": summary_matched["train_boxes"],
        "val_boxes": summary_matched["val_boxes"],
    }
]))


## Optional stop point
If you only want to prepare the datasets first, stop here.  
If the counts look good, continue to training below.


In [ ]:
# -----------------------
# Section 5 - Train YOLO detection on full dataset
# -----------------------
from ultralytics import YOLO

model_full = YOLO(YOLO_MODEL)
results_full = model_full.train(
    data=str(yaml_full),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    project=PROJECT_NAME,
    name="detect_full",
    exist_ok=True,
    verbose=True,
)


In [ ]:
# -----------------------
# Section 6 - Train YOLO detection on filtered dataset
# -----------------------
model_filtered = YOLO(YOLO_MODEL)
results_filtered = model_filtered.train(
    data=str(yaml_filtered),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    project=PROJECT_NAME,
    name="detect_filtered_balanced",
    exist_ok=True,
    verbose=True,
)


In [ ]:
# -----------------------
# Section 7 - Train YOLO detection on matched random dataset
# -----------------------
model_matched = YOLO(YOLO_MODEL)
results_matched = model_matched.train(
    data=str(yaml_matched),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    project=PROJECT_NAME,
    name="detect_full_matched_random",
    exist_ok=True,
    verbose=True,
)


In [ ]:
# -----------------------
# Section 8 - Validate all best checkpoints and compare
# -----------------------
def resolve_best_weights(train_results):
    save_dir = Path(train_results.save_dir)
    best_pt = save_dir / "weights" / "best.pt"
    if not best_pt.exists():
        raise FileNotFoundError(f"Could not find best weights at {best_pt}")
    return best_pt

best_full = resolve_best_weights(results_full)
best_filtered = resolve_best_weights(results_filtered)
best_matched = resolve_best_weights(results_matched)

from ultralytics import YOLO
best_model_full = YOLO(str(best_full))
best_model_filtered = YOLO(str(best_filtered))
best_model_matched = YOLO(str(best_matched))

metrics_full = best_model_full.val(data=str(yaml_full), device=DEVICE)
metrics_filtered = best_model_filtered.val(data=str(yaml_filtered), device=DEVICE)
metrics_matched = best_model_matched.val(data=str(yaml_matched), device=DEVICE)

def extract_metrics(metrics_obj):
    out = {}
    box = getattr(metrics_obj, "box", None)
    if box is not None:
        out["precision_B"] = getattr(box, "mp", None)
        out["recall_B"] = getattr(box, "mr", None)
        out["mAP50_B"] = getattr(box, "map50", None)
        out["mAP50_95_B"] = getattr(box, "map", None)
    else:
        for k in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
            try:
                out[k] = metrics_obj.results_dict.get(k)
            except Exception:
                pass
    return out

cmp_df = pd.DataFrame([
    {"dataset_condition": "full_unfiltered", **extract_metrics(metrics_full)},
    {"dataset_condition": "filtered_balanced", **extract_metrics(metrics_filtered)},
    {"dataset_condition": "full_matched_random", **extract_metrics(metrics_matched)},
])

print("[INFO] Best full weights:", best_full)
print("[INFO] Best filtered weights:", best_filtered)
print("[INFO] Best matched weights:", best_matched)
display(cmp_df)


In [ ]:
# -----------------------
# Section 9 - Save a compact comparison CSV
# -----------------------
comparison_path = BASE_DIR / "output" / "rd2_detection_comparison_metrics_matched.csv"
cmp_df.to_csv(comparison_path, index=False)
print("[INFO] Saved:", comparison_path)
